Step 0: Environment Check

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU count: {torch.cuda.device_count()}")

# Test CUDA functionality
if torch.cuda.is_available():
    x = torch.tensor([1.0, 2.0, 3.0]).cuda()
    print(f"CUDA tensor test: {x}")
    print("✓ PyTorch + CUDA working correctly!")
else:
    print("✗ CUDA not available!")

Step 1: Check Library Versions

In [ ]:
import transformers
import datasets
import peft

print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"PEFT version: {peft.__version__}")

In [ ]:
import bitsandbytes as bnb
import importlib.metadata

# Get version programmatically
version = importlib.metadata.version("bitsandbytes")
print(f"BitsandBytes version: {version}")

# Test 4-bit functionality
try:
    from bitsandbytes.nn import Linear4bit
    print("✓ Linear4bit available - QLoRA ready!")
except ImportError as e:
    print(f"✗ Linear4bit not available: {e}")


Step 2: Load Dataset & Tokenizer

In [ ]:
from datasets import load_dataset

dataset = load_dataset("glue", "sst2")
print(dataset["train"][0])


In [ ]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['sentence'], padding=True, truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)


Step 3: Load Model and Apply LoRA

In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)


Step 4: Training Arguments & Trainer

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sst2_results",
    eval_strategy="epoch",               
    per_device_train_batch_size=8,
    learning_rate=2e-5,
    num_train_epochs=3,
    fp16=True,
    logging_dir="./logs"
)


In [ ]:
model.gradient_checkpointing_enable()



In [ ]:
from transformers import Trainer, DataCollatorWithPadding

# Make sure label column is named "labels"
if "label" in tokenized_dataset["train"].column_names:
    tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

#  Create a Data Collator to pad batches dynamically
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create Trainer WITHOUT label_names
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

#  Start training
trainer.train()


In [ ]:
metrics = trainer.evaluate()
print(metrics)


In [ ]:
model.save_pretrained("./sst2_lora_model")
tokenizer.save_pretrained("./sst2_lora_model")


Step 5: Compute Metrics

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Function to compute metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)  # convert logits to predicted classes
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    return {"accuracy": acc, "f1": f1}

trainer_eval = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

metrics = trainer_eval.evaluate()
print(metrics)


In [ ]:
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import numpy as np


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)
    return {"accuracy": acc, "f1": f1}

def run_experiment(optimizer_name="adamw_torch", scheduler_type="linear"):
    print(f"\n🔹 Running experiment with {optimizer_name.upper()} + {scheduler_type} scheduler")

    training_args = TrainingArguments(
        output_dir=f"./results_{optimizer_name}_{scheduler_type}",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        fp16=True,
        save_total_limit=1,     # only keep latest checkpoint
        report_to="none"        # disable WandB/HF logging
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate()
    print("✅ Metrics:", metrics)
    return {
        "optimizer": optimizer_name,
        "scheduler": scheduler_type,
        "metrics": metrics
    }


experiments = []
experiments.append(run_experiment("adamw_torch", "linear"))
experiments.append(run_experiment("adafactor", "cosine"))
# skip Lion if not installed, or install via: pip install lion-pytorch
# experiments.append(run_experiment("lion", "cosine"))

# Compare all experiments
for exp in experiments:
    print(f"\nOptimizer: {exp['optimizer']}, Scheduler: {exp['scheduler']}")
    print(f"Accuracy: {exp['metrics']['eval_accuracy']:.4f}, F1: {exp['metrics']['eval_f1']:.4f}, Loss: {exp['metrics']['eval_loss']:.4f}")


In [ ]:
# save best version
model.save_pretrained("./sst2_lora_best_model")
tokenizer.save_pretrained("./sst2_lora_best_model")


In [ ]:
lr_scheduler_type="cosine"


In [ ]:
# Experiment Function Compatible with Older Transformers
def run_experiment(model, tokenized_dataset, optimizer_name="adamw_torch", scheduler_type="linear",
                   batch_size=8, lr=2e-5, epochs=3):
    """
    Runs one training experiment with specified optimizer, scheduler, batch size, learning rate, and epochs.
    
    Works with both older and newer Transformers versions (evaluation_strategy handled manually).
    """
    print(f"\n🔹 Running experiment with {optimizer_name.upper()} + {scheduler_type} scheduler")

    # TrainingArguments 
    # If your Transformers version is old, remove 'evaluation_strategy'
    training_args_kwargs = {
        "output_dir": f"./results_{optimizer_name}_{scheduler_type}",
        "num_train_epochs": epochs,
        "per_device_train_batch_size": batch_size,
        "learning_rate": lr,
        "fp16": True,                 # Mixed precision
        "save_total_limit": 1,        # Keep only latest checkpoint
        "logging_dir": "./logs",
        "report_to": "none"           # Disable WandB/HF logging
    }

    # Safe addition for new versions (optional)
    try:
        training_args_kwargs["evaluation_strategy"] = "epoch"
    except TypeError:
        pass  # Older versions ignore this

    training_args = TrainingArguments(**training_args_kwargs)

    # Trainer Setup
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    #Train the Model
    trainer.train()

    # Evaluate Metrics 
    metrics = trainer.evaluate()  # Works even if evaluation_strategy is ignored
    print("✅ Metrics:", metrics)

    # Save Model & Tokenizer
    model_path = f"./model_{optimizer_name}_{scheduler_type}"
    model.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    print(f"Model saved at {model_path}")

    # Return Metrics for Logging
    return {
        "optimizer": optimizer_name,
        "scheduler": scheduler_type,
        "batch_size": batch_size,
        "lr": lr,
        "epochs": epochs,
        "accuracy": metrics.get("eval_accuracy", None),
        "f1": metrics.get("eval_f1", None),
        "loss": metrics.get("eval_loss", None)
    }


In [ ]:
for exp in experiments:
    print(f"\nOptimizer: {exp['optimizer']}, Scheduler: {exp['scheduler']}")
    print(f"Accuracy: {exp['metrics']['eval_accuracy']:.4f}, "
          f"F1: {exp['metrics']['eval_f1']:.4f}, "
          f"Loss: {exp['metrics']['eval_loss']:.4f}")



In [ ]:
import shutil

# Find the best experiment (based on F1)
best_exp = max(experiments, key=lambda x: x['metrics']['eval_f1'])

#  Save Best Model 
# Correct path to the actual saved folder
best_model_src = f"./results_{best_exp['optimizer']}_{best_exp['scheduler']}"
best_model_dst = "./sst2_lora_best_model"

# If the destination exists, remove it first
if os.path.exists(best_model_dst):
    shutil.rmtree(best_model_dst)

shutil.copytree(best_model_src, best_model_dst)
print(f"✅ Best model saved at {best_model_dst}")



In [ ]:
import pandas as pd

df = pd.DataFrame(experiments)
df.to_csv("experiment_results.csv", index=False)
print("✅ Experiment results saved to experiment_results.csv")



In [ ]:
import pandas as pd
import ast

# Load CSV (correct path to your CSV file)
df = pd.read_csv("experiment_results.csv")  # replace with actual path if needed

# Convert 'metrics' column from string to dictionary
df['metrics'] = df['metrics'].apply(ast.literal_eval)

# Expand metrics dictionary into separate columns
metrics_df = pd.json_normalize(df['metrics'])
df_clean = pd.concat([df.drop(columns=['metrics']), metrics_df], axis=1)

# Optional: sort by evaluation accuracy descending
df_sorted = df_clean.sort_values(by='eval_accuracy', ascending=False).reset_index(drop=True)

# Display the comparison table
print("✅ Experiment Comparison Table:\n")
print(df_sorted[['optimizer', 'scheduler', 'eval_loss', 'eval_accuracy', 'eval_f1', 
                 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second']])

# Highlight best experiment
best_exp = df_sorted.iloc[0]
print("\n✅ Best Experiment:")
print(f"Optimizer: {best_exp['optimizer']}")
print(f"Scheduler: {best_exp['scheduler']}")
print(f"Metrics:")
print(f" - eval_loss: {best_exp['eval_loss']}")
print(f" - eval_accuracy: {best_exp['eval_accuracy']}")
print(f" - eval_f1: {best_exp['eval_f1']}")
print(f" - eval_runtime: {best_exp['eval_runtime']}")
print(f" - eval_samples_per_second: {best_exp['eval_samples_per_second']}")
print(f" - eval_steps_per_second: {best_exp['eval_steps_per_second']}")
